In [ ]:
import pandas as pd
import numpy as np

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# 🔹 Load dataset
df = pd.read_csv("/content/IMDB Dataset.csv")

# 🔹 Convert labels to 0/1
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# 🔹 Split data
X_train, X_test, y_train, y_test = train_test_split(
    df['review'], df['sentiment'], test_size=0.2, random_state=42
)

# 🔹 Tokenization
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# 🔹 Padding
max_len = 200
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

# 🔹 Model
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=64))
model.add(LSTM(64))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# 🔹 Train
model.fit(X_train_pad, y_train, epochs=5, batch_size=128, validation_split=0.2)

# 🔹 Evaluate
loss, acc = model.evaluate(X_test_pad, y_test)
print("Test Accuracy:", acc)

def predict_sentiment(text):
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len)

    prob = model.predict(padded)[0][0]

    if prob >= 0.5:
        return f"Positive  ({prob:.2f})"
    else:
        return f"Negative ({prob:.2f})"

print(predict_sentiment("This movie was fantastic!"))
print(predict_sentiment("I hated this film"))

Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 75s 293ms/step - accuracy: 0.8117 - loss: 0.4027 - val_accuracy: 0.8754 - val_loss: 0.2970
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 72s 289ms/step - accuracy: 0.8937 - loss: 0.2650 - val_accuracy: 0.8820 - val_loss: 0.2905
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 73s 290ms/step - accuracy: 0.9104 - loss: 0.2257 - val_accuracy: 0.8719 - val_loss: 0.2998
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 72s 289ms/step - accuracy: 0.9207 - loss: 0.2032 - val_accuracy: 0.8726 - val_loss: 0.3340
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 72s 285ms/step - accuracy: 0.9323 - loss: 0.1760 - val_accuracy: 0.8746 - val_loss: 0.3341
313/313 ━━━━━━━━━━━━━━━━━━━━ 9s 28ms/step - accuracy: 0.8820 - loss: 0.3143
Test Accuracy: 0.8820000290870667
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step
Positive 😊 (0.61)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Negative 😞 (0.22)


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 🔹 1. Toy dataset
input_texts = ["i love you", "i hate you", "hello", "how are you"]
target_texts = ["<start> je t aime <end>",
                "<start> je te deteste <end>",
                "<start> bonjour <end>",
                "<start> comment ca va <end>"]

# 🔹 2. Tokenization
input_tokenizer = Tokenizer()
input_tokenizer.fit_on_texts(input_texts)
input_seq = input_tokenizer.texts_to_sequences(input_texts)

target_tokenizer = Tokenizer(filters='')
target_tokenizer.fit_on_texts(target_texts)
target_seq = target_tokenizer.texts_to_sequences(target_texts)

# 🔹 3. Padding
max_input_len = max(len(seq) for seq in input_seq)
max_target_len = max(len(seq) for seq in target_seq)

input_seq = pad_sequences(input_seq, maxlen=max_input_len, padding='post')
target_seq = pad_sequences(target_seq, maxlen=max_target_len, padding='post')

# 🔹 4. Decoder input/output
decoder_input = target_seq[:, :-1]
decoder_output = target_seq[:, 1:]

# 🔹 5. Vocabulary sizes
vocab_in = len(input_tokenizer.word_index) + 1
vocab_out = len(target_tokenizer.word_index) + 1

latent_dim = 64

# 🔹 6. Encoder
encoder_inputs = Input(shape=(max_input_len,))
enc_emb = Embedding(vocab_in, latent_dim, mask_zero=True)(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)

encoder_states = [state_h, state_c]

# 🔹 7. Decoder
decoder_inputs = Input(shape=(max_target_len - 1,))
dec_emb = Embedding(vocab_out, latent_dim, mask_zero=True)(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(vocab_out, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# 🔹 8. Model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']   # ✅ stable accuracy
)

model.summary()

# 🔹 9. Train
history = model.fit(
    [input_seq, decoder_input],
    decoder_output.astype('int32'),   # ensures correct dtype
    epochs=300,
    validation_split=0.2,
    verbose=1
)

# 🔹 10. Evaluate
loss, acc = model.evaluate(
    [input_seq, decoder_input],
    decoder_output.astype('int32'),
    verbose=0
)

print(f"\nFinal Loss: {loss:.4f}")
print(f"Final Accuracy: {acc:.4f}")

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8       │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_9       │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_8         │ (None, 3, 64)     │        512 │ input_layer_8[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_8         │ (None, 3)         │          0 │ input_layer_8[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_9         │ (None, 4, 64)     │        768 │ input_layer_9[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_8 (LSTM)       │ [(None, 64),      │     33,024 │ embedding_8[0][0… │
│                     │ (None, 64),       │            │ not_equal_8[0][0] │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_9 (LSTM)       │ [(None, 4, 64),   │     33,024 │ embedding_9[0][0… │
│                     │ (None, 64),       │            │ lstm_8[0][1],     │
│                     │ (None, 64)]       │            │ lstm_8[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 4, 12)     │        780 │ lstm_9[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 68,108 (266.05 KB)

 Trainable params: 68,108 (266.05 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.0909 - loss: 2.4855 - val_accuracy: 0.5000 - val_loss: 2.4807
Epoch 2/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - accuracy: 0.2727 - loss: 2.4785 - val_accuracy: 0.2500 - val_loss: 2.4818
Epoch 3/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.3636 - loss: 2.4714 - val_accuracy: 0.2500 - val_loss: 2.4829
Epoch 4/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.5455 - loss: 2.4642 - val_accuracy: 0.2500 - val_loss: 2.4840
Epoch 5/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.5455 - loss: 2.4568 - val_accuracy: 0.2500 - val_loss: 2.4851
Epoch 6/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.5455 - loss: 2.4492 - val_accuracy: 0.2500 - val_loss: 2.4861
Epoch 7/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step - accuracy: 0.5455 - loss: 2.4412 - val_accuracy: 0.2500 - val_loss: 2.4871
Epoch 8/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.4545 - loss: 2.4328 - val_accuracy: 0.2500 - val_loss: 